# Orbit Wars — Behavior Cloning Pre-training

Bootstrap the policy network via supervised learning on **winning** moves from top-10%-rated Kaggle episodes. After this runs, `bc_init.pt` is saved — `train_ppo_kaggle.ipynb` auto-loads it and skips the random-exploration phase.

**Expected gain:** ~10× sample efficiency vs from-scratch PPO. The network starts already knowing what reasonable Orbit Wars moves look like.

## Data source: Bovard's curated top-10% replay datasets

Kaggle staff publishes a per-day dataset of the highest-rated Orbit Wars matchups:
- `bovard/orbit-wars-top10-episodes-YYYY-MM-DD` — one per UTC day, 2026-04-16 through 2026-05-04
- Already filtered to top 10% by `sum(both_agents_post_episode_rating)` — strong-bot-vs-strong-bot games
- Each ≈ 50-200 MB, replays in a `episodes.tar.gz`, manifest in `manifest.csv`

**Known data issue** (as of mid-May 2026): the earlier daily datasets have many missing 4P replay files compared to their manifests. The 2026-05-04 dataset is currently the most complete (2630 4P episodes, none missing). The notebook handles this by skipping manifest entries that aren't actually in the tar.

## How to run

You don't need to manually attach 19 datasets — the notebook **auto-downloads the whole series via the Kaggle CLI** by default. Just:

1. Confirm Kaggle API token works (it's automatic on a notebook running under your own account).
2. Attach an accelerator (GPU T4 is plenty for BC).
3. Save Version → Save & Run All. The notebook downloads each daily dataset, streams episodes out of its tar.gz in priority order, deletes the tar to free disk, and moves on. Output: `/kaggle/working/bc_init.pt`.
4. Then run `train_ppo_kaggle.ipynb` — it auto-copies `bc_init.pt` from `/kaggle/input/` and warm-starts PPO from it.

If you'd rather skip the CLI and use only datasets you've manually attached, set `SERIES_DATES = []` and rely on `+ Add Data` mounts.

## Architecture

Same encoder + network as `train_ppo_kaggle.ipynb` (94 K params). Trained with masked cross-entropy on `(target_planet, send_bin)` per my-planet, weighted by `my_mask` (only my planets contribute to loss).
Only **winning trajectories** are used — for each replay, extract the winner's moves and treat them as supervised labels. 4P winners are the highest-scoring player.

## 1. Setup

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle-environments>=1.28.0'])
print('installed')

In [ ]:
import os, json, math, time, random, gc, glob, subprocess
from collections import defaultdict, namedtuple
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from kaggle_environments import make
from kaggle_environments.envs.orbit_wars.orbit_wars import starter_agent

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  device={DEVICE}')

WORKING_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else '.'
REPLAY_DIR = os.path.join(WORKING_DIR, 'replays')
BC_CKPT = os.path.join(WORKING_DIR, 'bc_init.pt')
os.makedirs(REPLAY_DIR, exist_ok=True)
print(f'working dir: {WORKING_DIR}')
print(f'replays will go to: {REPLAY_DIR}')

## 2. Game constants + state encoder + network

Identical to `train_ppo_kaggle.ipynb` — must stay in sync. If you change one, change both.

In [ ]:
# --- engine constants ---
BOARD = 100.0
CENTER = 50.0
SUN_R = 10.0
ROTATION_LIMIT = 50.0
MAX_SPEED = 6.0
TOTAL_STEPS = 500
MAX_PLANETS = 48
MAX_FLEETS = 64
LAUNCH_CLEARANCE = 0.1
AIM_HORIZON = 120

PLANET_FEATURES = 12
FLEET_FEATURES = 8
GLOBAL_FEATURES = 8
SEND_FRACTIONS = [0.25, 0.5, 0.75, 1.0]
N_SEND_BINS = len(SEND_FRACTIONS)
NOOP_IDX = MAX_PLANETS


def fleet_speed(ships):
    if ships <= 1: return 1.0
    r = min(1.0, max(0.0, math.log(ships) / math.log(1000.0)))
    return 1.0 + (MAX_SPEED - 1.0) * (r ** 1.5)


def encode_state(obs, player_id):
    raw_planets = obs.get('planets', []) if isinstance(obs, dict) else getattr(obs, 'planets', [])
    raw_fleets  = obs.get('fleets',  []) if isinstance(obs, dict) else getattr(obs, 'fleets',  [])
    ang_vel     = obs.get('angular_velocity', 0.0) if isinstance(obs, dict) else getattr(obs, 'angular_velocity', 0.0)
    step        = obs.get('step', 0) if isinstance(obs, dict) else getattr(obs, 'step', 0)
    comet_ids   = set(obs.get('comet_planet_ids', []) if isinstance(obs, dict) else getattr(obs, 'comet_planet_ids', []))
    p_feat = np.zeros((MAX_PLANETS, PLANET_FEATURES), dtype=np.float32)
    p_mask = np.zeros((MAX_PLANETS,), dtype=np.float32)
    my_mask = np.zeros((MAX_PLANETS,), dtype=np.float32)
    for i, p in enumerate(raw_planets[:MAX_PLANETS]):
        pid, owner, x, y, radius, ships, prod = p
        dx, dy = x - CENTER, y - CENTER
        r = math.hypot(dx, dy); theta = math.atan2(dy, dx)
        is_static = 1.0 if r + radius >= ROTATION_LIMIT else 0.0
        is_comet = 1.0 if pid in comet_ids else 0.0
        ohot = [0.0, 0.0, 0.0, 0.0]
        if owner == player_id:
            ohot[0] = 1.0; my_mask[i] = 1.0
        elif owner == -1:
            ohot[2] = 1.0
        else:
            ohot[1] = 1.0
        p_feat[i] = [r/50.0, theta/math.pi, ang_vel/0.05 * (1.0 - is_static),
                     radius/3.0, ships/100.0, prod/5.0,
                     ohot[0], ohot[1], ohot[2], ohot[3],
                     is_static, is_comet]
        p_mask[i] = 1.0
    f_feat = np.zeros((MAX_FLEETS, FLEET_FEATURES), dtype=np.float32)
    f_mask = np.zeros((MAX_FLEETS,), dtype=np.float32)
    for i, f in enumerate(raw_fleets[:MAX_FLEETS]):
        fid, owner, x, y, angle, _, ships = f
        dx, dy = x - CENTER, y - CENTER
        r = math.hypot(dx, dy); theta = math.atan2(dy, dx)
        is_mine = 1.0 if owner == player_id else 0.0
        f_feat[i] = [r/50.0, theta/math.pi, math.cos(angle), math.sin(angle),
                     ships/100.0, fleet_speed(ships)/MAX_SPEED, is_mine, 1.0 - is_mine]
        f_mask[i] = 1.0
    my_ships = sum(p[5] for p in raw_planets if p[1] == player_id) + sum(f[6] for f in raw_fleets if f[1] == player_id)
    opp_ships = sum(p[5] for p in raw_planets if p[1] not in (-1, player_id)) + sum(f[6] for f in raw_fleets if f[1] not in (-1, player_id))
    total_ships = max(1, my_ships + opp_ships)
    my_planets = sum(1 for p in raw_planets if p[1] == player_id)
    opp_planets = sum(1 for p in raw_planets if p[1] not in (-1, player_id))
    g_feat = np.array([
        step / TOTAL_STEPS, ang_vel / 0.05,
        my_ships / total_ships, opp_ships / total_ships,
        my_planets / 12.0, opp_planets / 12.0,
        1.0 if comet_ids else 0.0, 1.0 if step > 100 and step < 400 else 0.0,
    ], dtype=np.float32)
    return p_feat, f_feat, g_feat, p_mask, f_mask, my_mask


# --- network architecture (must match PPO notebook) ---
HIDDEN = 64
N_HEADS = 4
N_LAYERS = 2

class PlanetEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.planet_proj = nn.Linear(PLANET_FEATURES, HIDDEN)
        self.fleet_proj = nn.Linear(FLEET_FEATURES, HIDDEN)
        self.global_proj = nn.Linear(GLOBAL_FEATURES, HIDDEN)
        enc_layer = nn.TransformerEncoderLayer(d_model=HIDDEN, nhead=N_HEADS,
                                                dim_feedforward=HIDDEN*2, batch_first=True,
                                                dropout=0.0, activation='gelu')
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=N_LAYERS)

    def forward(self, p_feat, f_feat, g_feat, p_mask, f_mask):
        ph = self.planet_proj(p_feat)
        pad_mask = (p_mask < 0.5)
        ph = self.transformer(ph, src_key_padding_mask=pad_mask)
        fh = self.fleet_proj(f_feat)
        f_w = f_mask.unsqueeze(-1)
        fh_pooled = (fh * f_w).sum(dim=1) / (f_w.sum(dim=1) + 1e-6)
        p_w = p_mask.unsqueeze(-1)
        ph_pooled = (ph * p_w).sum(dim=1) / (p_w.sum(dim=1) + 1e-6)
        gh = self.global_proj(g_feat)
        return ph, ph_pooled, fh_pooled, gh

class Actor(nn.Module):
    def __init__(self):
        super().__init__()
        self.query_proj = nn.Linear(HIDDEN, HIDDEN)
        self.key_proj = nn.Linear(HIDDEN, HIDDEN)
        self.noop_logit = nn.Parameter(torch.zeros(1))
        self.send_head = nn.Sequential(nn.Linear(HIDDEN, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, N_SEND_BINS))

    def forward(self, ph, p_mask, my_mask):
        B, MP, H = ph.shape
        Q = self.query_proj(ph); K = self.key_proj(ph)
        attn = torch.einsum('bmh,bnh->bmn', Q, K) / math.sqrt(H)
        eye = torch.eye(MP, device=attn.device).unsqueeze(0)
        attn = attn.masked_fill(eye.bool(), -1e9)
        pad = (p_mask < 0.5).unsqueeze(1).expand(-1, MP, -1)
        attn = attn.masked_fill(pad, -1e9)
        noop = self.noop_logit.expand(B, MP, 1)
        target_logits = torch.cat([attn, noop], dim=-1)
        send_logits = self.send_head(ph)
        return target_logits, send_logits

class Critic(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(nn.Linear(HIDDEN*3, HIDDEN), nn.GELU(), nn.Linear(HIDDEN, 1))

    def forward(self, ph_pooled, fh_pooled, gh):
        z = torch.cat([ph_pooled, fh_pooled, gh], dim=-1)
        return self.head(z).squeeze(-1)

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = PlanetEncoder(); self.actor = Actor(); self.critic = Critic()

    def forward(self, p_feat, f_feat, g_feat, p_mask, f_mask, my_mask):
        ph, php, fhp, gh = self.enc(p_feat, f_feat, g_feat, p_mask, f_mask)
        target_logits, send_logits = self.actor(ph, p_mask, my_mask)
        value = self.critic(php, fhp, gh)
        return target_logits, send_logits, value

net = ActorCritic().to(DEVICE)
print(f'net params: {sum(p.numel() for p in net.parameters()):,}')

## 3. Replay fetcher — auto-download the full top-10% series

Three sources, in order of preference:

1. **Datasets already attached** under `/kaggle/input/orbit-wars-top10-episodes-*` (from `+ Add Data`). Used directly — no download.
2. **Auto-download via Kaggle CLI** — for every date in `SERIES_DATES` that's *not* attached, run `kaggle datasets download bovard/orbit-wars-top10-episodes-<date>` into `/kaggle/working/ds_downloads/<date>/`. The tar.gz is deleted after extraction to keep disk usage bounded.
3. **API fallback** — if both above yield <50 replays, scrape episodes from your own submissions via `kaggle competitions episodes`.

For each tar (attached or downloaded), the notebook walks `manifest.csv` in priority order (highest combined rating first), streams the requested JSONs into `/kaggle/working/replays/`, and silently skips manifest entries not present in the tar.

In [ ]:
# === CONFIGURE ===
COMPETITION = 'orbit-wars'
MAX_TOTAL_REPLAYS = 3000           # cap. ~150 KB/replay -> 3000 = 450 MB on disk.

import datetime as _dt
def _date_range(start, end):
    s = _dt.date.fromisoformat(start); e = _dt.date.fromisoformat(end)
    return [(s + _dt.timedelta(days=i)).isoformat() for i in range((e - s).days + 1)]
SERIES_DATES = list(reversed(_date_range('2026-04-16', '2026-05-04')))  # latest first

ALLOW_CLI_DOWNLOAD = True
MAX_CLI_DOWNLOADS = 8

ALLOW_API_FALLBACK = True
N_RECENT_SUBMISSIONS_FALLBACK = 5
EPISODES_PER_SUBMISSION_FALLBACK = 100
# =================

import csv, tarfile, shutil, re

_DATE_RE = re.compile(r'(\d{4}-\d{2}-\d{2})')


def _parse_kaggle_csv(text, *id_keys):
    lines = [ln.strip() for ln in text.splitlines() if ln.strip() and not ln.startswith('Warning')]
    if len(lines) < 2: return []
    header = [h.strip() for h in lines[0].split(',')]
    id_col = next((header.index(k) for k in id_keys if k in header), None)
    if id_col is None: return []
    out = []
    for ln in lines[1:]:
        parts = ln.split(',')
        if len(parts) > id_col:
            try: out.append(int(parts[id_col]))
            except ValueError: pass
    return out


def _build_ep_index(root):
    """Walk `root` recursively and return {episode_id_str: absolute_path} for every <id>.json found."""
    idx = {}
    for dirpath, _, files in os.walk(root, followlinks=True):
        for fn in files:
            if not fn.endswith('.json'): continue
            ep_id = fn.rsplit('.', 1)[0]
            if not ep_id.isdigit(): continue
            # First match wins (avoid overwriting if the same id appears twice)
            idx.setdefault(ep_id, os.path.join(dirpath, fn))
    return idx


def _extract_from_source(manifest_path, tar_path, ep_index, downloaded, label):
    """Three modes (in priority order):
      1) manifest + tar  -> stream manifest-ordered entries out of tar
      2) manifest + ep_index (loose .json files) -> copy manifest-ordered files
      3) ep_index only -> bulk copy every .json under root (no priority)
    """
    if manifest_path and tar_path:
        return _extract_from_tar_manifest(tar_path, manifest_path, downloaded, label)
    if manifest_path and ep_index:
        return _extract_from_loose_manifest(manifest_path, ep_index, downloaded, label)
    if ep_index:
        return _bulk_copy_loose(ep_index, downloaded, label)
    print(f'  {label}: no tar, manifest, or loose JSONs — skipping')
    return downloaded


def _extract_from_tar_manifest(tar_path, manifest_path, downloaded, label):
    with open(manifest_path) as f:
        manifest = list(csv.DictReader(f))
    before = downloaded; skipped = 0
    with tarfile.open(tar_path, 'r:gz') as tar:
        try: present = set(tar.getnames())
        except Exception: present = None
        for row in manifest:
            if downloaded >= MAX_TOTAL_REPLAYS: break
            ep_id = row.get('episode_id') or row.get('id')
            if not ep_id: continue
            out_path = os.path.join(REPLAY_DIR, f'{ep_id}.json')
            if os.path.exists(out_path):
                downloaded += 1; continue
            member_name = f'episodes/{ep_id}.json'
            if present is not None and member_name not in present:
                skipped += 1; continue
            try:
                fin = tar.extractfile(member_name)
                if fin is None: continue
                with open(out_path, 'wb') as fout: fout.write(fin.read())
                downloaded += 1
                if downloaded % 250 == 0: print(f'    extracted {downloaded} replays so far')
            except (KeyError, tarfile.ReadError):
                skipped += 1
    print(f'  {label}: +{downloaded-before} from TAR (manifest={len(manifest)}, missing={skipped})')
    return downloaded


def _extract_from_loose_manifest(manifest_path, ep_index, downloaded, label):
    with open(manifest_path) as f:
        manifest = list(csv.DictReader(f))
    before = downloaded; missing = 0
    for row in manifest:
        if downloaded >= MAX_TOTAL_REPLAYS: break
        ep_id = row.get('episode_id') or row.get('id')
        if not ep_id: continue
        src = ep_index.get(str(ep_id))
        out_path = os.path.join(REPLAY_DIR, f'{ep_id}.json')
        if os.path.exists(out_path):
            downloaded += 1; continue
        if src is None:
            missing += 1; continue
        try:
            shutil.copy(src, out_path)
            downloaded += 1
            if downloaded % 250 == 0: print(f'    copied {downloaded} replays so far')
        except OSError:
            missing += 1
    print(f'  {label}: +{downloaded-before} from LOOSE (manifest={len(manifest)}, missing-on-disk={missing})')
    return downloaded


def _bulk_copy_loose(ep_index, downloaded, label):
    before = downloaded
    for ep_id, src in ep_index.items():
        if downloaded >= MAX_TOTAL_REPLAYS: break
        out_path = os.path.join(REPLAY_DIR, f'{ep_id}.json')
        if os.path.exists(out_path):
            downloaded += 1; continue
        try:
            shutil.copy(src, out_path)
            downloaded += 1
            if downloaded % 250 == 0: print(f'    copied {downloaded} replays so far')
        except OSError:
            pass
    print(f'  {label}: +{downloaded-before} from LOOSE-BULK (no manifest, {len(ep_index)} JSONs found)')
    return downloaded


def _extract_date(path):
    m = _DATE_RE.search(path)
    return m.group(1) if m else None


# ----- Cleanup stale downloads from previous runs -----
DS_DOWNLOAD_ROOT = os.path.join(WORKING_DIR, 'ds_downloads')
if os.path.isdir(DS_DOWNLOAD_ROOT):
    try:
        size_mb = sum(os.path.getsize(os.path.join(d, f))
                      for d, _, fs in os.walk(DS_DOWNLOAD_ROOT) for f in fs) / 1e6
        print(f'cleaning up stale ds_downloads/ ({size_mb:.0f} MB) from previous run...')
        shutil.rmtree(DS_DOWNLOAD_ROOT, ignore_errors=True)
    except OSError as e:
        print(f'  warning: could not fully clean ds_downloads/: {e}')


# ----- Step 1: discover attached datasets -----
# Strategy: find every dir directly containing a `manifest.csv`.
# That's the canonical dataset root for both tar-packaged and pre-extracted layouts.
INPUT_ROOT = '/kaggle/input'
sources = []  # list of (label, manifest_path_or_None, tar_path_or_None, ep_index_dict_or_None, date)

if os.path.isdir(INPUT_ROOT):
    manifest_dirs = []
    for dirpath, _, files in os.walk(INPUT_ROOT, followlinks=True):
        if 'manifest.csv' in files:
            manifest_dirs.append(dirpath)
    # Also catch tarball-only layouts (no manifest sibling, just episodes.tar.gz)
    extra_tar_dirs = []
    for dirpath, _, files in os.walk(INPUT_ROOT, followlinks=True):
        if 'episodes.tar.gz' in files and dirpath not in manifest_dirs:
            extra_tar_dirs.append(dirpath)

    for ds_root in sorted(set(manifest_dirs) | set(extra_tar_dirs), key=lambda p: _extract_date(p) or '0000-00-00', reverse=True):
        manifest_path = os.path.join(ds_root, 'manifest.csv')
        if not os.path.exists(manifest_path):
            # Check parent for a manifest
            alt = os.path.join(os.path.dirname(ds_root), 'manifest.csv')
            manifest_path = alt if os.path.exists(alt) else None
        tar_path = os.path.join(ds_root, 'episodes.tar.gz')
        if not os.path.exists(tar_path):
            tar_path = None
        # Build loose-files index from this dataset root if no tar
        ep_index = None
        if tar_path is None:
            ep_index = _build_ep_index(ds_root)
            if not ep_index: ep_index = None
        date = _extract_date(ds_root)
        label = f'date {date}' if date else os.path.basename(ds_root)
        sources.append((label, manifest_path, tar_path, ep_index, date))

# Diagnostic listing
print(f'\nDiscovered {len(sources)} dataset source(s) under {INPUT_ROOT}:')
for label, mp, tp, idx, date in sources:
    n_idx = len(idx) if idx else 0
    tar_sz = f'{os.path.getsize(tp)/1e6:.0f}MB' if tp else '—'
    print(f'  {label}: manifest={"yes" if mp else "no"}  tar={tar_sz}  loose_jsons={n_idx}')

# Extract — sorted by date desc
attached_dates = set()
downloaded = 0
for label, mp, tp, idx, date in sources:
    if downloaded >= MAX_TOTAL_REPLAYS: break
    if date: attached_dates.add(date)
    downloaded = _extract_from_source(mp, tp, idx, downloaded, label)


# ----- Step 2: auto-download missing dates -----
missing_dates = [d for d in SERIES_DATES if d not in attached_dates]
if not missing_dates:
    print(f'\nAll {len(SERIES_DATES)} series dates already covered — skipping CLI download.')
elif not ALLOW_CLI_DOWNLOAD:
    print(f'\nALLOW_CLI_DOWNLOAD=False; skipping {len(missing_dates)} missing date(s).')
elif downloaded >= MAX_TOTAL_REPLAYS:
    print(f'\nAlready at MAX_TOTAL_REPLAYS={MAX_TOTAL_REPLAYS}; skipping CLI download.')
else:
    os.makedirs(DS_DOWNLOAD_ROOT, exist_ok=True)
    n_dl = 0
    print(f'\nAuto-downloading up to {MAX_CLI_DOWNLOADS} of {len(missing_dates)} missing date(s) via Kaggle CLI '
          f'(have {downloaded}/{MAX_TOTAL_REPLAYS} replays)')
    for date in missing_dates:
        if downloaded >= MAX_TOTAL_REPLAYS: break
        if n_dl >= MAX_CLI_DOWNLOADS:
            print(f'\nHit MAX_CLI_DOWNLOADS ({MAX_CLI_DOWNLOADS}); stopping CLI path.')
            break
        slug = f'bovard/orbit-wars-top10-episodes-{date}'
        dest = os.path.join(DS_DOWNLOAD_ROOT, date)
        os.makedirs(dest, exist_ok=True)
        print(f'\n→ kaggle datasets download {slug}')
        try:
            subprocess.run(
                ['kaggle', 'datasets', 'download', slug, '-p', dest, '--unzip'],
                check=True, timeout=900,
                stdout=subprocess.DEVNULL, stderr=subprocess.PIPE,
            )
            n_dl += 1
        except subprocess.CalledProcessError as e:
            err = (e.stderr or b'').decode('utf-8', errors='ignore')[:300]
            print(f'  download failed: {err}')
            shutil.rmtree(dest, ignore_errors=True)
            continue
        except subprocess.TimeoutExpired:
            print(f'  download timed out (15 min)')
            shutil.rmtree(dest, ignore_errors=True)
            continue
        except OSError as e:
            print(f'  disk/OS error: {e}')
            shutil.rmtree(dest, ignore_errors=True)
            break
        # Use the same source-resolution logic
        mp = os.path.join(dest, 'manifest.csv') if os.path.exists(os.path.join(dest, 'manifest.csv')) else None
        tp = os.path.join(dest, 'episodes.tar.gz') if os.path.exists(os.path.join(dest, 'episodes.tar.gz')) else None
        idx = None if tp else _build_ep_index(dest)
        if idx is not None and not idx: idx = None
        downloaded = _extract_from_source(mp, tp, idx, downloaded, date)
        shutil.rmtree(dest, ignore_errors=True)


# ----- Step 3: API fallback -----
if downloaded < 50 and ALLOW_API_FALLBACK:
    print(f'\nOnly {downloaded} replays after datasets — falling back to your own submissions')
    try:
        out = subprocess.check_output(
            ['kaggle', 'competitions', 'submissions', COMPETITION, '-v'],
            stderr=subprocess.STDOUT, timeout=60,
        ).decode('utf-8', errors='ignore')
        sub_ids = _parse_kaggle_csv(out, 'submissionId', 'id')[:N_RECENT_SUBMISSIONS_FALLBACK]
        print(f'  found {len(sub_ids)} of your submissions: {sub_ids}')
    except Exception as e:
        print(f'  kaggle CLI failed: {e}'); sub_ids = []
    for sub_id in sub_ids:
        if downloaded >= MAX_TOTAL_REPLAYS: break
        try:
            out = subprocess.check_output(
                ['kaggle', 'competitions', 'episodes', str(sub_id), '-v'],
                stderr=subprocess.STDOUT, timeout=120,
            ).decode('utf-8', errors='ignore')
        except Exception: continue
        episode_ids = _parse_kaggle_csv(out, 'id', 'episodeId')[:EPISODES_PER_SUBMISSION_FALLBACK]
        for ep_id in episode_ids:
            if downloaded >= MAX_TOTAL_REPLAYS: break
            out_path = os.path.join(REPLAY_DIR, f'{ep_id}.json')
            if os.path.exists(out_path):
                downloaded += 1; continue
            try:
                subprocess.run(
                    ['kaggle', 'competitions', 'replay', str(ep_id), '-p', REPLAY_DIR],
                    timeout=60, check=True,
                    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                )
                downloaded += 1
            except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
                pass

shutil.rmtree(DS_DOWNLOAD_ROOT, ignore_errors=True)


n_local = len(glob.glob(os.path.join(REPLAY_DIR, '*.json')))
print(f'\n=== Total replay JSONs available: {n_local} ===')
if n_local < 50:
    print('  ⚠️  Fewer than 50 replays — BC will overfit.')
elif n_local < 500:
    print('  ⓘ  Modest dataset size — fine for a first BC run.')


## 4. Parse replays → (state, action) training pairs

For each replay, identify the winning player(s) and extract (observation, action) pairs from their trajectory. Convert engine moves `[from_id, angle, ships]` into our action labels `(target_planet_idx, send_bin)` per source planet.

In [ ]:
def fleet_trajectory_target(from_planet, angle, ships, planets, horizon=AIM_HORIZON):
    """Given a launched fleet, find which planet it hits first (continuous collision approximation).
    Returns target planet's id (in obs.planets) or None."""
    dirx, diry = math.cos(angle), math.sin(angle)
    speed = fleet_speed(ships)
    sr = from_planet[4]
    start_x = from_planet[2] + dirx * (sr + LAUNCH_CLEARANCE)
    start_y = from_planet[3] + diry * (sr + LAUNCH_CLEARANCE)
    best_t = 1e9; best_id = None
    for p in planets:
        if p[0] == from_planet[0]: continue
        dx = p[2] - start_x; dy = p[3] - start_y
        proj = dx * dirx + dy * diry
        if proj < 0: continue
        perp_sq = dx*dx + dy*dy - proj*proj
        if perp_sq >= p[4]*p[4]: continue
        hit_d = max(0.0, proj - math.sqrt(max(0.0, p[4]*p[4] - perp_sq)))
        t = hit_d / speed if speed > 0 else 1e9
        if t < best_t and t <= horizon:
            best_t = t; best_id = p[0]
    return best_id


def snap_send_bin(ships, source_ships):
    if source_ships <= 0: return 0
    frac = ships / source_ships
    return min(range(N_SEND_BINS), key=lambda i: abs(frac - SEND_FRACTIONS[i]))


def parse_replay(replay_json, only_winners=True):
    """Returns list of (state_dict, target_label, send_label) tuples.
    By default only extracts pairs from winning players."""
    pairs = []
    rewards = replay_json.get('rewards', [])
    if not rewards: return pairs
    # In 1v1, winner has reward 1, loser -1. Ties are 0 or both 1.
    # In 4P, only top-scorer is 1 (per orbit_wars interpreter). Use that.
    winners = [i for i, r in enumerate(rewards) if r is not None and r > 0]
    if only_winners and not winners: return pairs
    learners = winners if only_winners else list(range(len(rewards)))
    steps = replay_json.get('steps', [])
    for step_idx, step in enumerate(steps):
        if step_idx == 0: continue  # initial state, no action yet
        for p_idx in learners:
            if p_idx >= len(step): continue
            agent = step[p_idx]
            obs = agent.get('observation') or {}
            action = agent.get('action') or []
            if not obs.get('planets'): continue
            try:
                p_feat, f_feat, g_feat, p_mask, f_mask, my_mask = encode_state(obs, p_idx)
            except Exception:
                continue
            raw_planets = obs['planets']
            target_label = np.full(MAX_PLANETS, NOOP_IDX, dtype=np.int64)
            send_label = np.zeros(MAX_PLANETS, dtype=np.int64)
            # Build id->index map for obs.planets
            id_to_idx = {p[0]: i for i, p in enumerate(raw_planets[:MAX_PLANETS])}
            for move in action:
                if not isinstance(move, list) or len(move) != 3: continue
                from_id, angle, ships = move
                src_idx = id_to_idx.get(from_id)
                if src_idx is None: continue
                src = raw_planets[src_idx]
                if my_mask[src_idx] < 0.5: continue  # not actually our planet
                # Reconstruct source ships BEFORE the launch (engine subtracts after).
                # In replay, observation is from BEFORE action processed, so src[5] is pre-launch ships.
                tgt_id = fleet_trajectory_target(src, float(angle), int(ships), raw_planets[:MAX_PLANETS])
                if tgt_id is None: continue
                tgt_idx = id_to_idx.get(tgt_id)
                if tgt_idx is None: continue
                target_label[src_idx] = tgt_idx
                send_label[src_idx] = snap_send_bin(int(ships), int(src[5]))
            pairs.append({
                'p_feat': p_feat, 'f_feat': f_feat, 'g_feat': g_feat,
                'p_mask': p_mask, 'f_mask': f_mask, 'my_mask': my_mask,
                'target_label': target_label, 'send_label': send_label,
            })
    return pairs


# Parse all available replays
all_pairs = []
replay_files = sorted(glob.glob(os.path.join(REPLAY_DIR, '*.json')))
print(f'Parsing {len(replay_files)} replays...')
t0 = time.time()
for i, rf in enumerate(replay_files):
    try:
        with open(rf) as f:
            r = json.load(f)
        all_pairs.extend(parse_replay(r, only_winners=True))
    except Exception as e:
        print(f'  skip {os.path.basename(rf)}: {e}')
    if (i + 1) % 50 == 0:
        print(f'  parsed {i+1}/{len(replay_files)} ({len(all_pairs)} pairs so far, {time.time()-t0:.0f}s)')
print(f'\nTotal (state, action) pairs from winners: {len(all_pairs):,}')
if all_pairs:
    # Quick sanity stats
    n_with_action = sum(1 for p in all_pairs if (p['target_label'] < NOOP_IDX).any())
    n_my = float(sum(p['my_mask'].sum() for p in all_pairs))
    n_attacks = float(sum((p['target_label'] < NOOP_IDX).sum() for p in all_pairs))
    print(f'  pairs with at least one action: {n_with_action:,} ({100*n_with_action/len(all_pairs):.1f}%)')
    print(f'  attack-vs-noop ratio per my-planet: {n_attacks/max(1,n_my):.2%}')

## 5. Supervised training (masked cross-entropy)

Cross-entropy on target_label (target planet OR NOOP), cross-entropy on send_label, masked by `my_mask` (only count my-planets).

In [ ]:
BC_EPOCHS = 15
BC_BATCH_SIZE = 256
BC_LR = 3e-4
BC_VAL_FRACTION = 0.10

assert all_pairs, 'No training pairs — make sure replays are downloaded.'

# Pre-stack into numpy arrays (modest mem: ~10K pairs × ~5KB each = 50 MB)
def stack(field, dtype):
    arr = np.stack([p[field] for p in all_pairs], axis=0).astype(dtype)
    return torch.from_numpy(arr)

ds = {
    'p_feat': stack('p_feat', np.float32),
    'f_feat': stack('f_feat', np.float32),
    'g_feat': stack('g_feat', np.float32),
    'p_mask': stack('p_mask', np.float32),
    'f_mask': stack('f_mask', np.float32),
    'my_mask': stack('my_mask', np.float32),
    'target_label': stack('target_label', np.int64),
    'send_label': stack('send_label', np.int64),
}
N = ds['p_feat'].shape[0]
perm = np.random.RandomState(0).permutation(N)
n_val = int(N * BC_VAL_FRACTION)
val_idx = perm[:n_val]; train_idx = perm[n_val:]
print(f'train: {len(train_idx):,}  val: {len(val_idx):,}')

optimizer = torch.optim.Adam(net.parameters(), lr=BC_LR)

def run_epoch(indices, train=True):
    if train: net.train()
    else: net.eval()
    np.random.shuffle(indices) if train else None
    total_loss = 0; total_n = 0; total_target_acc = 0; total_send_acc = 0
    for start in range(0, len(indices), BC_BATCH_SIZE):
        mb = indices[start:start+BC_BATCH_SIZE]
        batch = {k: v[mb].to(DEVICE) for k, v in ds.items()}
        with torch.set_grad_enabled(train):
            target_logits, send_logits, _ = net(batch['p_feat'], batch['f_feat'], batch['g_feat'],
                                                  batch['p_mask'], batch['f_mask'], batch['my_mask'])
            # CE per (batch, planet); mask to my planets
            B, MP, _ = target_logits.shape
            tl_flat = target_logits.reshape(B*MP, -1)
            sl_flat = send_logits.reshape(B*MP, -1)
            tt_flat = batch['target_label'].reshape(-1)
            st_flat = batch['send_label'].reshape(-1)
            mask_flat = batch['my_mask'].reshape(-1)
            ce_target = F.cross_entropy(tl_flat, tt_flat, reduction='none')
            ce_send = F.cross_entropy(sl_flat, st_flat, reduction='none')
            n_eff = mask_flat.sum().clamp(min=1)
            loss_target = (ce_target * mask_flat).sum() / n_eff
            # send loss only counts cases where target != NOOP (otherwise send is meaningless)
            action_mask = mask_flat * (tt_flat != NOOP_IDX).float()
            n_action = action_mask.sum().clamp(min=1)
            loss_send = (ce_send * action_mask).sum() / n_action
            loss = loss_target + 0.5 * loss_send
        if train:
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            optimizer.step()
        with torch.no_grad():
            target_pred = tl_flat.argmax(-1)
            send_pred = sl_flat.argmax(-1)
            target_acc = ((target_pred == tt_flat).float() * mask_flat).sum() / n_eff
            send_acc = ((send_pred == st_flat).float() * action_mask).sum() / n_action
        total_loss += loss.item() * len(mb)
        total_n += len(mb)
        total_target_acc += target_acc.item() * len(mb)
        total_send_acc += send_acc.item() * len(mb)
    return total_loss / total_n, total_target_acc / total_n, total_send_acc / total_n

best_val_loss = float('inf')
for epoch in range(BC_EPOCHS):
    t0 = time.time()
    tr_loss, tr_t_acc, tr_s_acc = run_epoch(train_idx.copy(), train=True)
    val_loss, val_t_acc, val_s_acc = run_epoch(val_idx.copy(), train=False)
    print(f'epoch {epoch+1:2d}/{BC_EPOCHS}  '
          f'train: loss={tr_loss:.3f} tgt={tr_t_acc:.2%} send={tr_s_acc:.2%}  '
          f'val: loss={val_loss:.3f} tgt={val_t_acc:.2%} send={val_s_acc:.2%}  '
          f'[{time.time()-t0:.0f}s]')
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({'net': net.state_dict(), 'val_loss': val_loss, 'epoch': epoch+1}, BC_CKPT)
        print(f'  ★ saved best to {BC_CKPT}')

## 6. Sanity check — BC-trained agent vs starter

In [ ]:
# Reload best BC checkpoint
ck = torch.load(BC_CKPT, map_location=DEVICE, weights_only=False)
net.load_state_dict(ck['net'])
net.eval()
print(f'loaded best.pt (val_loss={ck["val_loss"]:.3f}, epoch {ck["epoch"]})')

def bc_agent_factory(net_ref):
    def agent(obs, config=None):
        player_id = obs.get('player', 0) if isinstance(obs, dict) else getattr(obs, 'player', 0)
        p_feat, f_feat, g_feat, p_mask, f_mask, my_mask = encode_state(obs, player_id)
        if my_mask.sum() < 0.5: return []
        pt = lambda a: torch.from_numpy(a).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            tl, sl, _ = net_ref(pt(p_feat), pt(f_feat), pt(g_feat), pt(p_mask), pt(f_mask), pt(my_mask))
        target_a = tl.squeeze(0).argmax(-1).cpu().numpy()
        send_a = sl.squeeze(0).argmax(-1).cpu().numpy()
        raw_planets = obs.get('planets', []) if isinstance(obs, dict) else getattr(obs, 'planets', [])
        moves = []
        for i in range(min(MAX_PLANETS, len(raw_planets))):
            if my_mask[i] < 0.5: continue
            tgt = int(target_a[i])
            if tgt == NOOP_IDX or tgt >= len(raw_planets) or tgt == i: continue
            src = raw_planets[i]; dst = raw_planets[tgt]
            frac = SEND_FRACTIONS[int(send_a[i])]
            ships = int(src[5] * frac)
            if ships < 1: continue
            angle = math.atan2(dst[3] - src[3], dst[2] - src[2])
            moves.append([src[0], angle, ships])
        return moves
    return agent

class CallableAgent:
    def __init__(self, fn): self.fn = fn
    def __call__(self, obs, config=None): return self.fn(obs)

starter_baseline = CallableAgent(starter_agent)
bc_agent = bc_agent_factory(net)

wins = 0; n = 12
for g in range(n):
    flip = g % 2 == 1
    env = make('orbit_wars', configuration={'episodeSteps': 300, 'actTimeout': 30, 'seed': 9000 + g}, debug=False)
    agents = [starter_baseline, bc_agent] if flip else [bc_agent, starter_baseline]
    state = env.run(agents)
    bc_seat = 1 if flip else 0
    r = state[-1][bc_seat].reward
    if r is not None and r > 0: wins += 1
    print(f'  g{g+1}: bc_seat={bc_seat} reward={r}')
print(f'\nBC agent vs starter: {wins}/{n} = {100*wins/n:.0f}%')
print('\nbc_init.pt is ready. Open train_ppo_kaggle.ipynb to fine-tune with PPO.')